In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")

!pip install -q ultralytics gradio folium gpxpy lapx
!git clone -q https://$GH_TOKEN@github.com/MaryemBourourou/potholemap.git /kaggle/working/repo
%cd /kaggle/working/repo
!ls src app weights

In [ ]:
import gdown
gdown.download(id="18ukTFl70ABm7Nx7WcCymF4V-uum8wDzO", output="/kaggle/working/clip1.mp4", quiet=False)
!ls -lh /kaggle/working/clip1.mp4

In [ ]:
!python src/pipeline.py --video /kaggle/working/clip1.mp4 --modele weights/best_v2.pt \
    --lat0 48.8566 --lon0 2.3522 --lat1 48.8606 --lon1 2.3376 --stride 2 --out /kaggle/working/sorties

In [ ]:
import pandas as pd
print(pd.read_csv("/kaggle/working/sorties/defauts.csv"))

In [ ]:
%%bash
ffmpeg -y -i /kaggle/working/sorties/annotee.mp4 -vcodec libx264 -pix_fmt yuv420p /kaggle/working/sorties/annotee_h264.mp4

In [ ]:
from base64 import b64encode
from IPython.display import HTML

with open("/kaggle/working/sorties/annotee_h264.mp4", "rb") as f:
  mp4 = b64encode(f.read()).decode()

HTML(
    f'<video width="720" controls><source src="data:video/mp4;base64,{mp4}"'
    ' type="video/mp4"></video>'
)

In [ ]:
import gdown

gdown.download(
    id="1P5B6PXIvMG0m9_Qp3gbS9VWNxhhybxGN",
    output="/kaggle/working/clip2.mp4",
    quiet=False,
)

In [ ]:
!ls -lh /kaggle/working/clip2.mp4

In [ ]:
%cd /kaggle/working/repo
!python src/pipeline.py \
    --video /kaggle/working/clip2.mp4 \
    --modele weights/best_v2.pt \
    --lat0 34.7406 --lon0 10.7603 \
    --lat1 34.7450 --lon1 10.7660 \
    --stride 2 \
    --out /kaggle/working/sorties_clip2

In [ ]:
import pandas as pd

print(pd.read_csv("/kaggle/working/sorties_clip2/defauts.csv"))

In [ ]:

  %%bash
  ffmpeg -y -i /kaggle/working/sorties_clip2/annotee.mp4 -vcodec libx264 -pix_fmt yuv420p /kaggle/working/sorties_clip2/annotee_h264.mp4

In [ ]:
from base64 import b64encode
from IPython.display import HTML

with open("/kaggle/working/sorties_clip2/annotee_h264.mp4", "rb") as f:
  mp4 = b64encode(f.read()).decode()

HTML(
    f'<video width="720" controls><source src="data:video/mp4;base64,{mp4}"'
    ' type="video/mp4"></video>'
)

In [ ]:
%cd /kaggle/working/repo
!python src/pipeline.py --help

In [ ]:
!grep -nE "conf|min_frames|min_conf|SEUILS_AIRE" src/pipeline.py

In [ ]:
%cd /kaggle/working/repo
!python src/pipeline.py \
    --video /kaggle/working/clip2.mp4 \
    --modele weights/best_v2.pt \
    --lat0 34.7406 --lon0 10.7603 \
    --lat1 34.7450 --lon1 10.7660 \
    --stride 2 \
    --conf 0.15 \
    --min_frames 5 \
    --min_conf 0.50 \
    --out /kaggle/working/essai_2

In [ ]:
import pandas as pd

print(pd.read_csv("/kaggle/working/essai_2/defauts.csv"))

In [ ]:
%cd /kaggle/working/repo
!python src/pipeline.py \
    --video /kaggle/working/clip2.mp4 \
    --modele weights/best_v2.pt \
    --lat0 34.7406 --lon0 10.7603 \
    --lat1 34.7450 --lon1 10.7660 \
    --stride 2 \
    --conf 0.25 \
    --min_frames 3 \
    --min_conf 0.40 \
    --out /kaggle/working/essai_3

In [ ]:
import pandas as pd

print(pd.read_csv("/kaggle/working/essai_3/defauts.csv"))

## Étape 4 — Comptage, essais de filtrage et gravité

### 4a. Comptage
| Clip | Nids-de-poule comptés à la main | Détections sans suivi | Défauts uniques avec suivi |
| :--- | :--- | :--- | :--- |
| **clip 1** | ~14 | 145 | 2 |
| **clip 2** | 36 | 13 | 0 (8 réparations) |

**Écart observé :** Sans suivi, les détections brutes sont soit très gonflées (145 sur clip 1), soit limitées (13 sur clip 2). Avec suivi, les doublons sont regroupés, mais il reste un écart avec le comptage manuel (2 retenus pour ~14 sur clip 1 ; 0 nid-de-poule retenu pour 36 sur clip 2, avec 8 zones réparées détectées) en raison de la compression vidéo, de la distance/angle de la caméra et des seuils de filtrage.

### 4b. Essais des paramètres (clip 2)
| Essai | `conf` | `min_frames` | `min_conf` | Défauts uniques | Vrais | Fausses alarmes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **1** | 0,15 | 5 | 0,40 | 8 | 7 | 1 |
| **2** | 0,15 | 5 | 0,50 | 6 | 6 | 0 |
| **3** | 0,25 | 3 | 0,40 | 9 | 8 | 1 |

**Limite notée :** Un même défaut peut parfois être découpé sur deux identifiants proches (ex. `#43` à 27.36 s et `#44` à 27.76 s).

### 4c. Gravité
Gravité relative : faible, moyenne ou élevée selon l'aire de la boîte (seuils 0,2 % et 0,6 % de l'image), abaissée d'un niveau si la confiance maximale est inférieure à 0,5. Ce n'est pas une mesure en centimètres : la taille apparente dépend de la distance et de l'angle de la caméra. Les zones réparées sont signalées séparément (« réparé »).

In [ ]:
import os

%cd /kaggle/working/repo
os.environ["SHARE"] = "1"
!python app/app.py

In [1]:
!git clone -q https://github.com/MaryemBourourou/potholemap.git /kaggle/working/repo_tmp
!git -C /kaggle/working/repo_tmp rev-parse --short HEAD

55882b3
